# Trading OS dataset exploration

This notebook is the exploration layer for Trading OS research. It loads a Java-produced bundle through the validated Python boundary, describes what data is actually available, and records questions for a human researcher.

It does not train a model, claim predictive signal, or replace the 0109B structured experiment. Historical candle time is not automatically historical knowledge available to Trading OS; STRICT_OBSERVED and known-at semantics remain authoritative.

## Environment and dataset selection

The default is the repository fixture. Replace `DATASET_DIR` with a Java export directory containing exactly `dataset-manifest.json` and `samples.jsonl`.

In [ ]:
from collections import Counter
from importlib.metadata import version, PackageNotFoundError
from pathlib import Path
import os
import platform
import sys

import matplotlib.pyplot as plt
import pandas as pd

def find_repository_root(start):
    for candidate in (start, *start.parents):
        if (candidate / 'research' / 'ml' / 'pyproject.toml').is_file():
            return candidate
    raise FileNotFoundError('Could not locate the Trading OS repository root')

REPOSITORY_ROOT = find_repository_root(Path.cwd())
sys.path.insert(0, str(REPOSITORY_ROOT / 'research' / 'ml' / 'src'))
from research_dataset.dataset import load_validated_bundle
from research_dataset.experiment import ReadinessPolicy, _readiness, readiness_population

DEFAULT_DATASET_DIR = REPOSITORY_ROOT / 'research' / 'ml' / 'tests' / 'fixtures' / 'java-generated'
DATASET_DIR = Path(os.environ.get('TRADING_OS_DATASET_DIR', DEFAULT_DATASET_DIR))
print('Python:', platform.python_version())
for package in ('numpy', 'pandas', 'matplotlib', 'jupyterlab'):
    try:
        print(f'{package}: {version(package)}')
    except PackageNotFoundError:
        print(f'{package}: unavailable')
print('Repository:', REPOSITORY_ROOT)
print('Dataset directory:', DATASET_DIR)

## Load and validate

Validation is mandatory. The notebook does not call `pandas.read_json` on `samples.jsonl`; it uses the immutable `ValidatedResearchDataset` boundary used by the structured experiments.

In [ ]:
dataset = load_validated_bundle(DATASET_DIR)
manifest = dataset.manifest
rows = list(dataset.rows)
print('Dataset valid.')
print('Validated rows:', len(rows))
print('Dataset fingerprint:', dataset.fingerprint)

## Identity, population, markets, and horizons

A chart without the dataset fingerprint, feature projection, and outcome definition is not sufficient research evidence.

In [ ]:
population = manifest.get('population', {})
identity = {
    'datasetFingerprint': manifest.get('datasetFingerprint'),
    'datasetDefinitionVersion': manifest.get('datasetDefinitionVersion'),
    'featureProjectionVersion': manifest.get('featureSchema', {}).get('version'),
    'outcomeDefinitionVersion': population.get('outcomeDefinitionVersion'),
    'sampleCount': manifest.get('sampleCount'),
    'eligibleCount': manifest.get('eligibleCount'),
    'excludedCount': manifest.get('excludedCount'),
    'marketsDeclared': population.get('marketIds', []),
    'horizonsDeclared': population.get('horizons', []),
}
display(pd.Series(identity, name='value').to_frame())

def count_table(values, name):
    counts = Counter(values)
    return pd.DataFrame([{name: key, 'count': value} for key, value in sorted(counts.items(), key=lambda item: str(item[0]))])

display(count_table([row.get('status') for row in rows], 'sampleStatus'))
display(count_table([row.get('marketId') for row in rows], 'marketId'))
display(count_table([row.get('horizon') for row in rows], 'horizon'))
display(count_table([(row.get('outcome') or {}).get('status') for row in rows], 'outcomeStatus'))

## Temporal coverage

This describes timestamps in the bundle. It does not prove that the underlying historical knowledge was available at those timestamps.

In [ ]:
cutoffs = sorted(pd.to_datetime([row['evaluationCutoff'] for row in rows], utc=True))
if len(cutoffs) > 0:
    print('First evaluationCutoff:', cutoffs[0])
    print('Last evaluationCutoff:', cutoffs[-1])
    print('Calendar duration:', cutoffs[-1] - cutoffs[0])
    by_day = pd.Series(cutoffs).dt.floor('D').value_counts().sort_index()
    by_day.plot(kind='line', marker='o', title='Samples by evaluation day')
    plt.ylabel('samples')
    plt.show()
else:
    print('No samples: temporal coverage cannot be plotted.')

## Canonical feature schema and missingness

The manifest schema is authoritative. Availability statuses are preserved as categorical facts; missingness is not converted to zero and is not confused with boolean false.

In [ ]:
definitions = list(manifest.get('featureSchema', {}).get('definitions', []))
schema_table = pd.DataFrame([{
    'feature': definition.get('name'),
    'group': definition.get('group'),
    'type': definition.get('dataType'),
    'unit': definition.get('unit'),
    'projectionVersion': manifest.get('featureSchema', {}).get('version'),
} for definition in definitions])
display(schema_table)

status_records = []
for definition in definitions:
    name = definition['name']
    statuses = Counter((row.get('features', {}).get(name) or {}).get('status', 'MISSING') for row in rows)
    for status, count in sorted(statuses.items()):
        status_records.append({'feature': name, 'type': definition.get('dataType'), 'status': status, 'count': count})
feature_statuses = pd.DataFrame(status_records)
display(feature_statuses)
if not feature_statuses.empty:
    present = feature_statuses.assign(present=feature_statuses['status'].eq('PRESENT')).groupby('feature')['present'].sum() / max(len(rows), 1)
    present.sort_values().plot(kind='barh', title='Feature PRESENT rate')
    plt.xlabel('rate')
    plt.show()

## Numeric, categorical, and boolean features

Decimal values are converted to float only for visualization. The validated dataset is never mutated. Select one feature at a time to keep the output readable.

In [ ]:
numeric_features = [item['name'] for item in definitions if item.get('dataType') == 'NUMERIC']
categorical_features = [item['name'] for item in definitions if item.get('dataType') == 'CATEGORICAL']
boolean_features = [item['name'] for item in definitions if item.get('dataType') == 'BOOLEAN']

def present_values(feature_name):
    return [row['features'][feature_name].get('value') for row in rows if (row.get('features', {}).get(feature_name) or {}).get('status') == 'PRESENT']

if numeric_features:
    selected_numeric = numeric_features[0]
    values = pd.Series([float(value) for value in present_values(selected_numeric)])
    if not values.empty:
        display(values.describe(percentiles=[.25, .5, .75]).rename(selected_numeric).to_frame())
        values.plot(kind='hist', bins=20, title=f'{selected_numeric} distribution')
        plt.show()
else:
    print('No numeric feature is available.')

if categorical_features:
    selected_categorical = categorical_features[0]
    statuses = Counter((row.get('features', {}).get(selected_categorical) or {}).get('status', 'MISSING') for row in rows)
    values = Counter(present_values(selected_categorical))
    display(pd.Series(statuses, name='statusCount').to_frame())
    display(pd.Series(values, name='valueCount').to_frame())
else:
    print('No categorical feature is available.')

if boolean_features:
    selected_boolean = boolean_features[0]
    states = Counter((row.get('features', {}).get(selected_boolean) or {}).get('status', 'MISSING') if (row.get('features', {}).get(selected_boolean) or {}).get('status') != 'PRESENT' else str(row['features'][selected_boolean].get('value')).lower() for row in rows)
    display(pd.Series(states, name='count').to_frame())
else:
    print('No boolean feature is available.')

## Objective outcomes and temporal target view

Outcomes are inspected independently from model features. Only fields that actually exist in the validated contract are displayed. A visual pattern or exploratory correlation is not out-of-sample predictive evidence.

In [ ]:
available_outcomes = [row.get('outcome', {}) for row in rows if (row.get('outcome') or {}).get('status') == 'AVAILABLE']
outcome_fields = sorted({field for outcome in available_outcomes for field in outcome})
print('Available outcome fields:', outcome_fields)
target_rows = [row for row in rows if (row.get('outcome') or {}).get('status') == 'AVAILABLE' and 'forwardReturn' in (row.get('outcome') or {})]
if target_rows:
    target_values = pd.Series([float(row['outcome']['forwardReturn']) for row in target_rows])
    display(target_values.describe(percentiles=[.25, .5, .75]).rename('forwardReturn').to_frame())
    target_values.plot(kind='hist', bins=20, title='forwardReturn distribution')
    plt.show()
    selected_market, selected_horizon = Counter((row.get('marketId'), row.get('horizon')) for row in target_rows).most_common(1)[0][0]
    target_frame = pd.DataFrame({'evaluationCutoff': pd.to_datetime([row['evaluationCutoff'] for row in target_rows if row.get('marketId') == selected_market and row.get('horizon') == selected_horizon], utc=True), 'forwardReturn': [float(row['outcome']['forwardReturn']) for row in target_rows if row.get('marketId') == selected_market and row.get('horizon') == selected_horizon]})
    target_frame.plot(x='evaluationCutoff', y='forwardReturn', title=f'{selected_market} {selected_horizon} forwardReturn')
    plt.show()
else:
    print('No available forwardReturn observations: target exploration skipped.')

## Data readiness

This reuses the 0109B readiness function. It does not create a competing notebook-specific gate and it never trains a model.

In [ ]:
READINESS_HORIZON = 'PT4H'
declared_market_ids = tuple(population.get('marketIds', []))
requested_market_id = os.environ.get('TRADING_OS_READINESS_MARKET_ID')
if requested_market_id and requested_market_id not in declared_market_ids:
    raise ValueError('TRADING_OS_READINESS_MARKET_ID must be declared in population.marketIds')
READINESS_MARKET_ID = requested_market_id or (declared_market_ids[0] if len(declared_market_ids) == 1 else None)
if READINESS_MARKET_ID is None:
    eligible_available, readiness_counts, readiness = [], {'validatedRows': len(rows), 'marketRows': 0, 'horizonRows': 0, 'eligibleRows': 0, 'availableOutcomeRows': 0, 'experimentRows': 0}, None
    print('Data readiness: blocked (an explicit single market identity is required)')
else:
    eligible_available, readiness_counts = readiness_population(dataset, horizon=READINESS_HORIZON, market_id=READINESS_MARKET_ID)
    readiness = _readiness(eligible_available, readiness_counts, ReadinessPolicy())
print('Framework readiness: available')
if readiness is not None:
    print('Data readiness:', 'ready' if readiness.ready else 'blocked')
print('Readiness market:', READINESS_MARKET_ID or 'explicit identity required')
print('Readiness horizon:', READINESS_HORIZON)
print('Eligible samples:', len(eligible_available))
if readiness is not None:
    print('Strict observed coverage days:', readiness.facts['strictObservedCoverageDays'])
    print('Approximate non-overlapping 4h windows:', readiness.facts['approxNonOverlapping4hWindows'])
    print('Blocking reasons:', list(readiness.reasons) or 'none')
if not eligible_available:
    print('There is currently insufficient strictly observed data for statistical exploration.')

## Research Notes

Use these prompts during interactive exploration. Do not pre-answer them without evidence from the selected validated dataset.

- Which features have sufficient coverage?
- Which features appear nearly constant?
- Are categorical states heavily imbalanced?
- Are there suspicious temporal gaps?
- Which horizons have enough observations?
- Does missingness change over time?
- Are there apparent market regimes worth investigating later?

Notebook outputs are exploratory. Clear runtime-specific outputs before committing the notebook. A promising visual observation must become a hypothesis and then a reproducible structured Python experiment before supporting a research conclusion.